# CDCN CVPR 2020 · Kiểm tra MTCNN và PRNet trên OULU-NPU
Dùng runtime **CPU** và chạy các cell theo thứ tự. Mẫu nhỏ lấy 3 video và
9 khung ở mỗi tập Train và Dev của Protocol 1. Mỗi tập có video thật,
ảnh in và phát lại. Kết quả là ảnh scene, bbox, depth JPEG và JSON kiểm tra.
Notebook dành cho bước chuẩn bị dữ liệu trước huấn luyện.

In [ ]:
#@title 1. Kết nối Drive và chọn thư mục OULU-NPU
from pathlib import Path
import os
import shutil
import sys
import time
from google.colab import drive

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/face-pad/datasets/OULU-NPU")
WORK = Path("/content/cdcn-prnet-smoke")
RUN = Path("/content/drive/MyDrive/face-pad/cdcn-prnet-smoke-p1") / str(time.time_ns())
for name, size in {"Protocols.tar": 417792, "Train_files.tar": 37753662976,
                   "Dev_files.tar": 28151992320}.items():
    source = DATA / name
    if not source.is_file() or source.stat().st_size != size:
        raise ValueError(f"Kiểm tra đường dẫn và dung lượng: {source}")
RUN.mkdir(parents=True)
print("Thư mục kết quả:", RUN)

In [ ]:
#@title 2. Nạp code và cài môi trường tách riêng
BUNDLE_SHA256 = '0cbca94f03da7aa4523a34f81494c26ecd40fdbdf6af2cb75095045d56ba97a4'
BUNDLE = (
    'H4sIAAAAAAAC/+08aW/jRpb5LGD+Qy3nw1LdFK3DdruNaADHdne86cNrO9kdOAZRIosW2xTJJSnbisb72/cdVTx0uNO7gwDBWkDa'
    'pTpevXr3e1VKkfs7gVJZKH3lZTLY8bwoiUrPc7PFd/+kTx8++7u79Bc+q3/7/Tdv6jb2D0bD0eA70f/uD/jMi1LmsP13/z8/lmWd'
    'APvfAfvF+dGJyFWhZO5PRSb9O3mrXJjwl85fOp53r/IiShPPE2Nh9d2B24eB714+f+5Psar/WZ7eghAUf5z+7+0NRqv6Pxrtv+j/'
    'H6T/5yqP0iDyhWG9CNNcJGmpJml6J76kk0LIJBBxmmYF2oNOmKcz4XnhvJznCgxCNMvSvIRJsEiWYCSKDs/JZDmNo4mZcA5fO7pd'
    'zCewnw/bmZ5ymisZRMlt1RHNVKfT8WMJOJ1r5A47Aj6BCoXxVHah4tARsZyo2BEloBCPP6WJcsQcxsdWVKpZYTkiSkqV38PgqN9l'
    'KPjBxa5eS20CoNsIQDfNarB+zZ14kxp4G24AaMCC/upuRenNs0CWOEhdKISlCuArHtqdpUDJNIl8u9teWpRphpMMqdzTe5WUMKui'
    'CsPVNMH9G2eNQuoR/zKu0atHV7HGP+uDbeTXsK3wmAGvwH9oROB8ZROTJH147qwqlllB5MCJvRaNqkkazQrlasBP50kJI6EllnRE'
    '52lnWfMWvoplxd4nC8lSj4qoQNkXKEGAR6EECHyFVynR+1lNitLRxBi683mSAE8s0pYGRPzaF98zwt83Rtq0j4JYtU/coHVrJqNh'
    'w/n+IU6vjsR/L+3Gdj1mu3hVkXGHeg7dfvhU0HFpq+9XBPuVGLZ2qXgBNKCtklQk6kGTF23EEuFosDXzcgVmIYEl18tauZ5ugOZI'
    'qaclrX8CeAa9pW4wpCWc7smqBcmbQkBSTpQs6ZANIXqYRnAMZFalG+6DjHheda5um8pZDgM8w4hoxbiuI8J4XkzHV/lcNWTZ8xQC'
    '06amAXATMC2mq8BaSsT621LkK2rZsPhWlWOaVZ8cNFkqUJTtsFg77DUu4JTWQR4bJlM9+l65yBS3gFZzaJY5hAITiP5WjSTRt1Cl'
    'vRmBL2mUNIY2kSaUwLCAJNBsrVUsBFNeTNep1ukg2sAfL05vb1Vg++lsBvrkiFeVxYcRR/gPgTb6KrmnlkYf/NW7NH+QeQCCO5tE'
    'CQicD4ITiHReZnPwMilwQQlfxaypGYbA+b2i3jSPbqMElEPleZqT80OgsCUwD92ZDc2u6XMzmYOkuLO7IMpt/lLQSZDCEShzetdg'
    'oT4LQLouytwGzndJraAB/sQM33RqcobWh/T2UCxhrydrXb4eonJKeKSZSmzr0UJq+CnK19ial2HvwOoKWeDJncqh2pqMlXvcb7pH'
    'Aln7avecQFdcQLLDf0x1+A8tfQDgx80lZ+enzibL0nQtZQAUbq66vDr5/PMViKN6LA0JV87iMFeKsZWrLAap5dMhhLbOl/nicA0B'
    'pHQM4oCkxiUuY364EVMYcB9yiCRsXNLdOoc4Ym8eZxbiejxKMLaszRaidmMB+gPCjcxaewpokMpK8YMs1Ck1Ie5aRx5UjQBkaRzb'
    'XXRun9acfo0gTAQhmIHEw0nXMdpIx2ol4YguHfm/t75aI9zg8RVPPn3MohzM/3bQdxFi/5Wt18dzGRWq6auRpO1taIq4AI8EuJyi'
    'OIGSLdlloc5iUEY6QNxY4r9Prng3B2PB9sNoY/fPmQsXfh5lZbHjB37iZXmiSq+YpXfqn5j+fSX/G/UHg/5a/jd4yf/+qPzvQvmQ'
    'sZX53C/F8cnxp15RLiCu+nh1/OnTzvnFJ1WCiURPCWmYKjk7lGICUVwAykHSAoFA+bszQ9OX34KHBP3U3/3i3jSnssC00XzlP5s6'
    'ILQoJYTH0ox8KdLEtGeNbDMttiak4AkB1/O0iB5b+el0Xkbx9my1WNSJq8xDCGyqr2qWtb5jGmvaEPLUJPDvh6aZzGfZAr1XkjGm'
    'hR9lC7cAhCNMSnjWv0/B8JCV0gl2s3TjVvm7OZ3+3ukAF0+vvOPPHz+eXWECE/qDYaj21CAc7O4PJ4GSe8Pd/tvJcDAK+n448A/8'
    'cN/fm1idk9Pzqx8bK/39YNAPh2qyLydv94dyMhrshgcDJXel7PdlOAjejoLJBFb+x+nZ+x+vvMsfj3Dd3ttgNDqYhH1/OJgM+hN1'
    'MBiFgzfhwW5/d+jDoNybhKPBgdo7ODjY2wvD4WTo93cH/nBvMnk7kFbn7NPJ6X8acP7k4GDwdhj64Zvd3cmBGgJaSu7L4f7BwA/e'
    '9sNBfyiHwds3b+RkAMHsqD/cnwThrq/eHOzBEqtz+fHzT6fe+dHxT0fvTy8xneJwsVRJkeZhnD6Mx0N3sO8OLUePhL07lctC9/cb'
    '/UUczcbjARVFde+s9JME+/p1HzEZ+4b77q7pBEbfRWUvmkGIPB733eFuvYCEgAC/cQemEyMw/76XLcppmvQgR4AkrACsdt0B7naw'
    'DxO7+nyXHz5f0eH6jhg54k23c34GXVV/Oc9iZecygfD8oGvi7WIqbdQUHQeS/8PvOq7MJxxogc1Qcna4mnJo7XVRB7wgulUF5AE0'
    '1YETTeVwb9/qulP1qMfMrnEqA28GDjYubPJEDkAEG5OrXCOSFi7EmFGeJtfW1Tvv58tT78Pp+6Pjv3s/nV4cXVo3KBoDa23u8c8n'
    'R94vZ5dnP3w49U5Ofzk7PtWTeW6luIb1LgS4cGD3foDnLEOaRQpHbDULyERydvVXcfVOoFz8ayF4cTSJ4qhc0LFUDrFEpnywnpRV'
    'QPah6+sw7JNhdAkMWBUXSDAHjl6DYHkkcG6sbqW/8OB7LBcqLwj32ghyy+N1dr2syP31pd1v2MdN0nwm4+g3wvDZXSsheH77FYi0'
    'qqvT0xLoA65lzGS1A3Uf+WpsHZ//fNg3aIOg4x3EbS6zKYpv6L7Hpt3If1pzXFl4IFtyHoOc1ZL6V/EBuEK8yKIEU0LI+oKI9icg'
    'Etg56EVJ7+rdcIWffG63TnRBMShMB4kVO8KqIEEIZXVWg3nUIkyXPcxr7LXszNWZjN0KUq01AWWx/DXZJLlJmQN7mN40z49/TUh4'
    'N0wLczlTD2l+J2q37BU+KLrlfAWHVSWpsQk9NIgre+vOZ7apT4+W0uP4utgBAyxaHdDTrDMYxo3Zu7ofSSKvoG3XzGisaMl+PeGG'
    'mKi/tSYT06KkUHmJdhRzdeJ2t4a5lhmpR+VjmpyBEbSR+XodWlU4Dg5jraMWFs/Dhue1sxgincyiyqVffGqNk7nEUsTFJzsCUQfF'
    'HL+TcaEIdBg9jjdhG2JJI17BuDpplmZ2v7s2pklGww2yOZRQ8nQ21xsZEUDqN/XIyWlWaL0xJh5VByOuYqcxtVKhFjlrJZosSlXY'
    '3c20ZcgrhNU+ylgbhylYTb5VCdiuEvxWjYQpQtVhsk1a42yDog1NfjuB80GM5/r35XEaY2rJ67Dv+POHzxfeD+8vhhfvf2DcMJAr'
    'sPCuwbrc8KjfBnA8DfJYrHlSb8P5UiL7C9bwOI21yIyKIApoOsMCwxZSoYTWPTpi4YgHRyAjZvLRJpiOuFOLcSxnk0AKvDw5pH+v'
    'rUn6aN1cD2/Eq1bH6Karm8xXiKQp0keYSBZXR854AgfiW1fmuVzY17D7o3iN2wMSC2hNb7SM3oOmRUwLhnALaanptKsNjOPIGoJk'
    'm2lOY2ksk2Am87vmWgf540I4guXPqPiogndRXKq8UYzRsmLXZOo6vJ0WCT9PM48SocIGAjhCgyt8GSuI2kZaElp0honUOVXR7RRi'
    'nIcoIPxp7fXh8EbHEz9BaE8eigMIiCvAWubkN0Wao55JEAIYLMGB9LI0SkpwZo9gDggjmUegglxn5Oo1YKFbC4wIkfb0z0O3C8o3'
    'dISNTKB/ptzFeMo49BB3/EuI4ulABh54lfk6rVbEKiy1QIG5xDTMBTxBJA0ioqehan7nSAlcESU2kWPzqtftVXwVtnWXhdllqudP'
    '0rIEa8rbGOpvWve6vQ70jU70t7FGFHwN7v23sQb5nBJCLggxA9EoINVD/lst+QK3CC4PIDoaHtoz6sNtHd60Cs1VjCbhPgpUimFy'
    'WqZ+iteVWRyxMCmvkDOI6tkPNEN4nadyFF8vtfJDjufx3Ud0r5rHwVs63e1CwJJLv0QQdmid6/XFjml5g50lofHklo8lxTIysLtg'
    'xdD/2Rzi9Iro1hw/fUANv7apLIoVNzh4AgbSBSsbZXaXq+G6PI0DVBUv7gkwqDqg59KGWFMtcD6yCqcbCEaXPidKEMkIopL4vgWS'
    'J7Wjg0V9NcDKhSV5msK0xvsCvIFmXaJlnl7G6A8cMQDM7V1HDLs3iEODC3y/oScxOv401cbt2iZwjolZHSF9tJR88PYYlanbm1Mh'
    'hhbgIG4B6rird2EJgU2WTx1T7t68G1GVcapZv4HyyK92yBBGKg7oIMhC9AmMObZwBTMTOWRbnqURa5RjbQZw3b9xNKzrQd0Ev4LX'
    'qttotLHWzXKEywZdvOa29RGpY3OBeU1nQ4uBzKICrIM/PRRLPMiTtV5gZiJf4zCGj7zZ2qwJyOtdp3mfuqkG3cIA9i7ArJvqGihe'
    'OQc/rPVLLIkmdI1NRIEW7/3UNi2MnzYdahZpw0GhkiFkZTs4vNZ+E7vjtCzGjTqCJqAvMx3X/ILAjmWGxT67CsG0dW5GxOZGHsUE'
    'lqNTtikIOjr3zi8+n3vvLo4+nnrHn3/+dNWIU6lijyu/F4P+V2nmU5WRAh28wlCCoi2GAKSjqHGVkSjTjpjzW4PrG5Sy5kXCX8VH'
    'lABOFOcTMMriRN3XHjlJZ3Q7iOYmUK54RzuenaAthTQ/Ub2JBNhuS62QrqgdRN/2qQw8kFbwQjTxlbCZBjs7QISV7ACg0SE9CPJQ'
    'Q6mOo4E45OcMxNdiD7wkQ3oNurBZfZrAkCibNQZzxyiZq/UbK8kXxC3Wnn++ZPZeOjX8HqCwtjy90zNQvAAU+49NeCKf0zt0xHJS'
    'bJaoetsu7GY27qLX7rt733iwrRdfFPXVEehahvCKlWnz1VWeRnjS1SCSlnJkuvUSza4l32lUhbvfeC58EITI825oMBubYyyKJOYZ'
    'Ad3WwwwI4Oegxwe/05paXPgCBYPUA/DsFRBnKUEgNOVmMrM2SwPVR1TgzSqTE824z7bcL9mtpYn/jJA8h+bnS40j6EXMaRJD14j9'
    '2/np+w2YATajocYnV0X0m7IZNR3pNLDmXO/s48Xp0Yn3/uLo75fHRx9Ou9fAejA29ggc9mjY3Yo+7eTKZGF3kRNNb0ZhCs4BfmRl'
    'ZtPU7rcKQNPA6eudhq3TpctcYaMQEwUGRwk6Mj5Z6azf4qsZvbtiY4tfn7yl0b3D/ih4stYXoTx4uhqg32TssE/CmgCl4IUFzQom'
    'tGELhk6rSRQ2sWk7WBjcDpS4PzjZDLbG9xufe2zC7f8AQosISx6/TUAXXKPX1aak+3tUIGzoAAGD2KKC9GQ9cwKarSsxDcl3y1RX'
    'Z7rPkRATE6+Yh2H0CCodSMwbGCIVSK1fE4tfFgFIPB/mSI8Q6b8WOLQOGr2WK4PAriz++qsA8PmuzCAJCuylRQERzLMOa1nAewoU'
    'FOjTsZHFmgcdJmiy0DvjBPjzldctlsEFppsmljTZNUOn8dhfAaOv5+czbzSEVRhMsYGALiAzgIQ4MUo8fkPm/abytEIYLffgqfuV'
    '2FSHjUihzsZCIbvlWEFUU93bzCQ+/DrURb28oAqgud11j/LbOdjs8pxG7EDxiwMIWseeF6S+KczxSmSdJ/US2+r18HIXrDw6nzFe'
    'zz47mcuLv3c2Zmk9ztIsDIYJJ6sowcZ54MSxc6ribGxBHp0Knc5R4C1267RsaD27CXGECjQRysuz+2S5QgsgqDTBklKIDMhJMvr8'
    'PiQbzaNjMfK/5viupmE/tiw2Ndj/7Xo2r19dDUsKugMhIPQHwRR2VWfBN3AFWgFpXipxCkjdxN2qv/EIk+EpduXFPMviBdAQ/Fgu'
    'WIDQdxrpqCqMJoFv5Md3aoGIZ6kjqqgFM2vbMuQlRPTlZPNSHfN/q0FGmme+O6J5i9700fiOLsLEqPEoyp8q/07fddjX1m2E+1q9'
    'Y4ur3Igeqnqu7nt0chz9EcIL66bxWK5rKiDtXIp2A0tgTvfVnGoJJHkShBJgJGbzooQwQCwNgKZnMDS9hjWYDfN2XH5SusJX0w+d'
    '8AnwZgfaxKSd4d6+h/9BfBH6CbZ4GclDj54G9dKQGvqON+Krap5G+X7j0QFmWtXgip+JgDGPVpeWVO8Knivj6YCIX8MQObjgCst6'
    'gKkIojCsb1fplMzB5z07l+jqCl31JtOi+0mu4GOgRxtbDcnhvAJ/jNC4NW9KZ0sCKwWrFanxqwDSKxNeU9Kln6Pqqe2MvKWGvz8p'
    '5/dzNdCm5GzNomjmhixKxz6cmVB4/BVELOYc+zfMRRLBldl27rEaTDUYifLK+2X8Unwl8eDXnRYbhbOPR+/xccnlpeXw6XCQl2PB'
    'Gp019kDsTim5l6QJumue0t3ydJyx79R1C3xPPc+5lAjaf80FNqor4J1g46WHYb1xRx65Iy5ONio8vP7536JQCYPCaDSN1hVCRBt0'
    'ou6bAlrVH9sV69rAAz2rGrILW1pVIYrmNGqoNQV0KdqE9U1gGLxTcczDAjVDXGFONeUQf8t0Ryc5+uVMzNRsgnflcJ5lcwddLdrM'
    'jLa+VqDF1dEFaSxxEtbGKrH57Gi1uVW9U6bFjvh1veDQLNU3cVov1zvVgzP3SuEtscwXJ+B58QYR80a8hYf+FQVRMxYEYlBpMFyr'
    'LjFpkDJrlwNVzYoLNq1HdDavIwJuT255TlTQjQIluFwwJivtxukDxItkoi1X3kdWPQHTtYSLaNVBvjHxxWc3sXnDj/TBCy+Cjv+s'
    'TedHIRsuQvgQ/Bwqnee+chi0fir1oJ9KQdi75WU5vzMEM5AtEGA6+WIbQLjIEbvilRj0h+bPhkLJBH+vIM2da6PMS5g4zSo10e6m'
    'rWkb675b0iY4OqZNzS27m4m1ajs5nHXxgWYMOR6TR277hcIXIEGy+gOppmA2MEAxaCJ0uDWN0lD1S35ExQ3ms6xonmd7akn3gEn7'
    '7Cid2Mk024KuttIbc86KKY4hEXSubbMhcWMpmydxlNxtevpu9AJigFl6r+xqo/W51U9+9W8HGyZL9AiZClp30+oV27p8bv3TTnP4'
    'ybiJf4jlqrPpaT906A7w92XP/1ZCm5SvWIN2stvijRbsFm+oiQ9KMISA79WtTO00rCeSRr5bxJp+iiiv0Us/9iNHXUDkbC7FxmOr'
    'e92/0T5V+Qii/TD1Rl/n0OsfSFZqIeEQwDInwNqCbmJxgkwInkEH5dDHkbCnX2AeNmLl1cKDmUphcr2gipQdrEXoS99qFIPtZ7x7'
    'd32T2sNjmWLF6esKC6lDhC9IScPWgBjKWvqW7nDD43BXE4F0gG8ozZ2mWf60BrjAMDRAuKuCib/OZNHkzHHV0tGdHZZpZuCJyeC1'
    'a1oNu8N8xVgwgPRgPKxfHBhePpOUhNYl3Q5GCT6MilUJh1+adSa81mEpvuv3ODY18kWxKSknCFK3Nj+rP8CL8GeDSC/8Pz+AL/Y8'
    'lG3Psxg1rv68/I8gXj4vn5fPy+fl8/J5+bx8Xj4vnz/B538A8OYxoABQAAA='
)
import base64
import hashlib
import io
import tarfile
bundle = base64.b64decode(BUNDLE)
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Code đi kèm không khớp SHA-256")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        dest = (WORK / member.name).resolve()
        if not member.isfile() or not dest.is_relative_to(WORK):
            raise ValueError(f"Invalid bundle member: {member.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.extractfile(member).read())
(RUN / f"source-{BUNDLE_SHA256[:12]}.tar.gz").write_bytes(bundle)
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import Progress, run_logged

def run(label, *args):
    run_logged(args, label=label, log=RUN / f"{label}-{time.time_ns()}.log",
               env={**os.environ, "PYTHONPATH": str(WORK / "src"), "TF_ENABLE_ONEDNN_OPTS": "0"})

run("install-uv", sys.executable, "-m", "pip", "install", "uv")
VENV = WORK / "venv"
PY = VENV / "bin/python"
if not PY.is_file():
    run("create-venv", sys.executable, "-m", "uv", "venv", "--python", "3.12.10", VENV)
run("install-prnet", sys.executable, "-m", "uv", "pip", "install", "--python", PY,
    *('tensorflow==2.16.2', 'tf-keras==2.16.0', 'tf-slim==1.1.0', 'mtcnn==1.0.0', 'numpy==1.26.4', 'scikit-image==0.24.0', 'scipy==1.17.1', 'opencv-python-headless==4.11.0.86'))

In [ ]:
#@title 3. Tải code và trọng số PRNet công khai
PRNET = WORK / "PRNet"
RENDERER = WORK / "PRNet-Depth-Generation"
for path, repo, commit in [
    (PRNET, "https://github.com/yfeng95/PRNet.git", "fc12fe5e1f1462bdea52409b213d0cf1c8cf6c5b"),
    (RENDERER, "https://github.com/clks-wzz/PRNet-Depth-Generation.git", "c6d10f2eb6ab962ab314f81ea4aa00af1d93dbbb"),
]:
    if not path.exists():
        run(f"clone-{path.name}", "git", "clone", repo, path)
    run(f"checkout-{path.name}", "git", "-C", path, "checkout", "--detach", commit)
WEIGHT = PRNET / "Data/net-data/256_256_resfcn256_weight.data-00000-of-00001"
def weight_matches(path):
    if not path.is_file():
        return False
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest() == "59d338bf0c21b10be813f17f84042c9d3a5bf318e588855ff2b2c041c25bb91a"

if not weight_matches(WEIGHT):
    partial = WEIGHT.with_suffix(".part")
    run("download-prnet", "curl", "-LfsS", "--retry", "2", "-o", partial,
        "https://drive.usercontent.google.com/download?id=1UoE-XuW1SDLUjZmJPkIZ1MLxvQFgmTFH&export=download&confirm=t")
    if not weight_matches(partial):
        raise ValueError("Tệp tải về không khớp SHA-256 của checkpoint PRNet")
    partial.replace(WEIGHT)
run("verify-prnet", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
    "--image", PRNET / "TestImages/0.jpg", "--prnet", PRNET, "--renderer", RENDERER,
    "--output", WORK / f"public-{time.time_ns()}")

In [ ]:
#@title 4. Dựng dữ liệu mẫu Train/Dev và lưu lên Drive
OUT = WORK / f"oulu-{time.time_ns()}"
WIDE_SAMPLE = False  # True để thêm phone 4/session 2, đọc sâu hơn trong TAR
run("oulu-prnet-smoke", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
    "--data", DATA, "--prnet", PRNET, "--renderer", RENDERER, "--output", OUT,
    *(["--wide-sample"] if WIDE_SAMPLE else []))
with Progress("Save smoke assets to Drive"):
    local_zip = Path(shutil.make_archive(str(OUT), "zip", OUT))
    temporary = RUN / (local_zip.name + ".part")
    shutil.copyfile(local_zip, temporary)
    temporary.replace(RUN / local_zip.name)
    shutil.copyfile(OUT / "smoke_summary.json", RUN / "smoke_summary.json")
print("CDCN_PRNET_SMOKE_READY", RUN)